In [8]:
!pip install mmengine

     ---------------------------------------- 0.0/46.8 kB ? eta -:--:--
     -------- ------------------------------- 10.2/46.8 kB ? eta -:--:--
     -------------------------------------- 46.8/46.8 kB 589.1 kB/s eta 0:00:00
   ---------------------------------------- 0.0/452.7 kB ? eta -:--:--
   ------------- -------------------------- 153.6/452.7 kB 4.6 MB/s eta 0:00:01
   ------------------ --------------------- 215.0/452.7 kB 2.6 MB/s eta 0:00:01
   ---------------------------------- ----- 389.1/452.7 kB 3.0 MB/s eta 0:00:01
   ---------------------------------------- 452.7/452.7 kB 2.8 MB/s eta 0:00:00
   ---------------------------------------- 0.0/39.5 MB ? eta -:--:--
   ---------------------------------------- 0.2/39.5 MB 3.6 MB/s eta 0:00:12
   ---------------------------------------- 0.4/39.5 MB 3.7 MB/s eta 0:00:11
    --------------------------------------- 0.5/39.5 MB 3.7 MB/s eta 0:00:11
    --------------------------------------- 0.7/39.5 MB 3.5 MB/s eta 0:00:12
    --

In [1]:
import torch
import sys
import os
from torch.utils.data import DataLoader
from sklearn.metrics import precision_recall_curve, f1_score, roc_auc_score
from transformers import AutoTokenizer
import numpy as np
import pandas as pd
from torch import nn

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))  # Going one level up from the current directory
sys.path.append(project_root)

# Import dataset classes from the provided files
from datasets.hint.hint import HINTDataset
#from datasets.ctod.ctod import CTODDataset
from mmf.early_fusion import EarlyFusion
from mmf.late_fusion import LateFusion
from mmf.middle_fusion import MiddleFusion
from mmcto.layers.sparse_moe import SparseMOELayer
from mmcto import MMCTO

In [2]:
# Placeholder function to calculate PR, F1, and ROC
def calculate_metrics(y_true, y_pred):
    # Compute Precision-Recall curve
    precision, recall, _ = precision_recall_curve(y_true, y_pred)
    pr = 2 * (precision * recall) / (precision + recall + 1e-8)
    
    # Compute F1 score
    f1 = f1_score(y_true, y_pred > 0.5)
    
    # Compute ROC AUC
    roc = roc_auc_score(y_true, y_pred)
    
    return pr.mean(), f1, roc

In [3]:
# Define the LIFTED model class (generic, no method-specific conditionals)
class LIFTEDModel(nn.Module):
    def __init__(self, vocab_size=28996, model_dim=768, fusion_type="early", num_experts=4):
        super(LIFTEDModel, self).__init__()
        self.model_dim = model_dim
        self.num_experts = num_experts
        
        # Define the embedding layer
        self.embedding = torch.nn.Embedding(vocab_size, model_dim)
        
        # Use the fusion model (Early, Late, Middle)
        if fusion_type == "early":
            self.fusion = EarlyFusion(encoders={}, model_dim=model_dim)
        elif fusion_type == "late":
            self.fusion = LateFusion(encoders={}, model_dim=model_dim)
        elif fusion_type == "middle":
            self.fusion = MiddleFusion(encoders={}, model_dim=model_dim)

        # Add the Sparse Mixture of Experts (SMOE) Layer
        self.smoe = SparseMOELayer(
            expert_cfg={"type": "some_expert_layer"},  # You can define the actual expert layer
            num_experts=num_experts,
            input_dim=model_dim,
        )
        
        # Transformer model for different LIFTED variants
        self.transformer = AutoModel.from_pretrained("bert-base-cased")
        
        # Linear layer to get the final output
        self.fc = torch.nn.Linear(model_dim, 1)

    def forward(self, input_ids, attention_mask=None):
        # Embedding layer
        embedded = self.embedding(input_ids)
        
        # Apply transformer encoding (BERT)
        transformer_output = self.transformer(input_ids=input_ids, attention_mask=attention_mask)
        
        # Take the hidden states from the transformer output
        hidden_state = transformer_output.last_hidden_state[:, 0, :]  # [batch_size, hidden_dim]
        
        # Apply the fusion model (Early, Late, or Middle fusion)
        fusion_output = self.fusion(hidden_state, attention_mask)
        
        # Apply the SMOE layer to the fused output
        smoe_output = self.smoe(fusion_output)
        
        # Pass through the final linear layer
        logits = self.fc(smoe_output['logits'])
        return logits


In [4]:
# Function to load data for different methods and features (use your dataset classes)
def load_data_for_method(dataset, feature, phase, batch_size=32):
    # Using the dataset classes you provided, instantiate the dataset
    data_prefix = {
        "data_path": "Data Group Part/Data/clinical-trial-outcome-prediction/Processed/hint/",
        "table_path": "text_description",
        "summarization_path": "brief_summary",
        "drug_description_path": "drugbank/druginfo_description.json",
        "criteria_path": "criteria",
    }
    
    # Adjust to use your dataset classes HINTDataset or CTODDataset
    dataset_class = HINTDataset if dataset == "HINT" else CTODDataset
    ann_file_name = f"{dataset.lower()}_{feature}_{phase}"
    
    dataset_instance = dataset_class(
        ann_file_name=ann_file_name,  # Change this based on phase, method, feature
        data_prefix=data_prefix
    )
    
    # Create data loaders
    train_loader = DataLoader(dataset_instance, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(dataset_instance, batch_size=batch_size, shuffle=False)
    
    return train_loader, val_loader

In [5]:
# Function to train and evaluate the model
def train_and_evaluate(model, train_loader, val_loader, phases, feature, dataset):
    results = {}
    
    for phase in phases:
        # Train the model for the current phase
        model.train()  # Assuming model.train() handles phase-specific training
        for batch in train_loader:
            input_ids, attention_mask, labels = batch
            outputs = model(input_ids, attention_mask)
            # Compute loss (example: Binary Cross-Entropy)
            loss = torch.nn.BCEWithLogitsLoss()(outputs.squeeze(), labels.float())
            loss.backward()
            # Optimizer step (e.g., Adam) would go here
            
        # Evaluate the model
        model.eval()
        y_true = []
        y_pred = []
        
        for batch in val_loader:
            input_ids, attention_mask, labels = batch
            with torch.no_grad():
                outputs = model(input_ids, attention_mask)
            y_true.append(labels)
            y_pred.append(outputs.squeeze())
        
        # Convert to numpy arrays
        y_true = torch.cat(y_true).cpu().numpy()
        y_pred = torch.cat(y_pred).cpu().numpy()
        
        # Calculate metrics
        pr, f1, roc = calculate_metrics(y_true, y_pred)
        
        # Store the results for the phase
        results[phase] = {
            "PR": pr,
            "F1": f1,
            "ROC": roc
        }
    
    return results

In [6]:
# Loop over features and datasets
features = ['summarization', 'drugs', 'disease', 'smiles', 'criteria']
phases = ['Phase I', 'Phase II', 'Phase III']
datasets = ['HINT']

for dataset in datasets:
    for feature in features:
        print(f"Training and evaluating with {feature} feature on {dataset} dataset...")
            
        # Define your model
        model = LIFTEDModel(fusion_type="early", num_experts=4)
            
        # Load your data for each feature combination
        train_loader, val_loader = load_data_for_method(dataset, feature, 'train')
            
        # Call train and evaluate function
        results = train_and_evaluate(model, train_loader, val_loader, phases, feature, dataset)
            
        # Print results
        print(f"Results for {feature} feature on {dataset} dataset:")
        for phase, metrics in results.items():
            print(f"{phase}: PR: {metrics['PR']:.2f}, F1: {metrics['F1']:.2f}, ROC: {metrics['ROC']:.2f}")


Training and evaluating with summarization feature on HINT dataset...


KeyError: 'some_expert_layer'